# 08 — Benchmark de frameworks (solução)

**Objetivo:** medir, na sua máquina, o tempo das mesmas 6 operações em pandas / Polars / DuckDB / PySpark. Os defaults rodam em ~30s num laptop. Aumente `linhas` para escalar.

Operações cronometradas (mesma lista da página `frameworks-processamento.md`):

1. Carga de Parquet
2. Filtro `value > 50`
3. Agregação `group by category` (sum, mean)
4. Sort por `value desc`
5. Join contra sample 1%
6. Write Parquet

> **Exercício:** preencha os `# TODO`s para medir cada framework.

In [ ]:
# Parametros usados pelo Papermill.
run_date = None
project_root = None
linhas = 2_000_000
colunas_cardinalidade = 50
repeticoes = 3
incluir_pyspark = True

from pathlib import Path
import sys

for candidato in [Path.cwd(), *Path.cwd().parents]:
    caminhos_common = [
        candidato / "_common.py",
        candidato / "aula02" / "_common.py",
        candidato / "exercicios" / "_common.py",
        candidato / "sessao-02-data-architecture" / "_common.py",
        candidato / "sessao-02-data-architecture" / "exercicios" / "_common.py",
    ]
    encontrado = next((p for p in caminhos_common if p.exists()), None)
    if encontrado:
        sys.path.insert(0, str(encontrado.parent))
        break
else:
    raise FileNotFoundError("Nao encontrei _common.py a partir do diretorio atual.")

from _common import configurar_paths
paths = configurar_paths(project_root)
globals().update(paths)

BENCH_DIR = DATA_DIR / "benchmark"
BENCH_DIR.mkdir(parents=True, exist_ok=True)
DATASET_PATH = BENCH_DIR / "dataset.parquet"
SAMPLE_PATH = BENCH_DIR / "sample.parquet"
RESULTS_CSV = BENCH_DIR / "resultados.csv"

print(f"linhas = {linhas:,} | repeticoes = {repeticoes} | incluir_pyspark = {incluir_pyspark}")
print(f"BENCH_DIR = {BENCH_DIR}")


## Geração do dataset sintético

In [ ]:
import time
import psutil
import os
import gc
import numpy as np
import pandas as pd

if DATASET_PATH.exists() and SAMPLE_PATH.exists():
    print(f"Reusando dataset: {DATASET_PATH}")
    print(f"Reusando sample:  {SAMPLE_PATH}")
else:
    rng = np.random.default_rng(42)
    df_sintetico = pd.DataFrame({
        "id": np.arange(linhas, dtype=np.int64),
        "category": rng.integers(0, colunas_cardinalidade, size=linhas).astype(str),
        "value": rng.normal(loc=50, scale=20, size=linhas),
        "timestamp": pd.Timestamp("2026-01-01") + pd.to_timedelta(rng.integers(0, 90 * 24 * 60, size=linhas), unit="m"),
        "payload": rng.choice(["alpha", "beta", "gamma", "delta"], size=linhas),
    })
    df_sintetico.to_parquet(DATASET_PATH, index=False)
    df_sintetico.sample(frac=0.01, random_state=42)[["id", "category"]].to_parquet(SAMPLE_PATH, index=False)
    print(f"Dataset criado: {DATASET_PATH}")
    print(f"Sample criado:  {SAMPLE_PATH}")


## Helpers

In [ ]:
from contextlib import contextmanager

process = psutil.Process(os.getpid())
resultados: list[dict] = []

@contextmanager
def medir(framework, operacao, repeticao):
    gc.collect()
    rss_ini = process.memory_info().rss
    t0 = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - t0
        rss_fim = process.memory_info().rss
        resultados.append({
            "framework": framework,
            "operacao": operacao,
            "repeticao": repeticao,
            "tempo_s": elapsed,
            "memoria_mb_delta": (rss_fim - rss_ini) / (1024 ** 2),
        })


## Benchmarks por framework

In [ ]:
def bench_pandas():
    for rep in range(repeticoes):
        with medir("pandas", "load_parquet", rep):
            df = pd.read_parquet(DATASET_PATH)
        with medir("pandas", "filter", rep):
            filtered = df[df["value"] > 50]
        with medir("pandas", "groupby", rep):
            grouped = df.groupby("category")["value"].agg(["sum", "mean"]).reset_index()
        with medir("pandas", "sort", rep):
            sorted_df = df.sort_values("value", ascending=False)
        with medir("pandas", "join", rep):
            sample = pd.read_parquet(SAMPLE_PATH)
            joined = df.merge(sample, on=["id", "category"], how="inner")
        with medir("pandas", "write_parquet", rep):
            filtered.to_parquet(BENCH_DIR / f"pandas_write_{rep}.parquet", index=False)
        del df, filtered, grouped, sorted_df, sample, joined
        gc.collect()


def bench_polars():
    try:
        import polars as pl
    except ImportError as exc:
        print(f"Polars indisponivel: {exc}")
        return

    for rep in range(repeticoes):
        with medir("polars", "load_parquet", rep):
            df = pl.read_parquet(DATASET_PATH)
        with medir("polars", "filter", rep):
            filtered = df.filter(pl.col("value") > 50)
        with medir("polars", "groupby", rep):
            grouped = df.group_by("category").agg(
                pl.col("value").sum().alias("sum"),
                pl.col("value").mean().alias("mean"),
            )
        with medir("polars", "sort", rep):
            sorted_df = df.sort("value", descending=True)
        with medir("polars", "join", rep):
            sample = pl.read_parquet(SAMPLE_PATH)
            joined = df.join(sample, on=["id", "category"], how="inner")
        with medir("polars", "write_parquet", rep):
            filtered.write_parquet(BENCH_DIR / f"polars_write_{rep}.parquet")
        del df, filtered, grouped, sorted_df, sample, joined
        gc.collect()


def bench_duckdb():
    try:
        import duckdb
    except ImportError as exc:
        print(f"DuckDB indisponivel: {exc}")
        return

    dataset = str(DATASET_PATH).replace("\\", "/").replace("'", "''")
    sample = str(SAMPLE_PATH).replace("\\", "/").replace("'", "''")
    con = duckdb.connect()
    try:
        for rep in range(repeticoes):
            with medir("duckdb", "load_parquet", rep):
                loaded = con.execute(f"SELECT * FROM read_parquet('{dataset}')").fetch_arrow_table()
            with medir("duckdb", "filter", rep):
                filtered = con.execute(f"SELECT * FROM read_parquet('{dataset}') WHERE value > 50").fetch_arrow_table()
            with medir("duckdb", "groupby", rep):
                grouped = con.execute(f"""
                    SELECT category, SUM(value) AS sum, AVG(value) AS mean
                    FROM read_parquet('{dataset}')
                    GROUP BY category
                """).fetch_arrow_table()
            with medir("duckdb", "sort", rep):
                sorted_df = con.execute(f"SELECT * FROM read_parquet('{dataset}') ORDER BY value DESC").fetch_arrow_table()
            with medir("duckdb", "join", rep):
                joined = con.execute(f"""
                    SELECT d.*
                    FROM read_parquet('{dataset}') d
                    INNER JOIN read_parquet('{sample}') s USING (id, category)
                """).fetch_arrow_table()
            out = str(BENCH_DIR / f"duckdb_write_{rep}.parquet").replace("\\", "/").replace("'", "''")
            with medir("duckdb", "write_parquet", rep):
                con.execute(f"COPY (SELECT * FROM read_parquet('{dataset}') WHERE value > 50) TO '{out}' (FORMAT PARQUET)")
            del loaded, filtered, grouped, sorted_df, joined
            gc.collect()
    finally:
        con.close()


def bench_pyspark():
    if not incluir_pyspark:
        print("PySpark desativado por parametro.")
        return
    try:
        from pyspark.sql import SparkSession, functions as F
    except ImportError as exc:
        print(f"PySpark indisponivel: {exc}")
        return

    spark = SparkSession.builder.master("local[*]").appName("benchmark-frameworks").getOrCreate()
    spark.sparkContext.setLogLevel("WARN")
    try:
        for rep in range(repeticoes):
            with medir("pyspark", "load_parquet", rep):
                df = spark.read.parquet(str(DATASET_PATH))
                df.count()
            with medir("pyspark", "filter", rep):
                filtered = df.filter(F.col("value") > 50)
                filtered.count()
            with medir("pyspark", "groupby", rep):
                grouped = df.groupBy("category").agg(F.sum("value").alias("sum"), F.mean("value").alias("mean"))
                grouped.count()
            with medir("pyspark", "sort", rep):
                sorted_df = df.orderBy(F.col("value").desc())
                sorted_df.count()
            with medir("pyspark", "join", rep):
                sample = spark.read.parquet(str(SAMPLE_PATH))
                joined = df.join(sample, on=["id", "category"], how="inner")
                joined.count()
            with medir("pyspark", "write_parquet", rep):
                filtered.write.mode("overwrite").parquet(str(BENCH_DIR / f"pyspark_write_{rep}.parquet"))
    finally:
        spark.stop()


bench_pandas()
bench_polars()
bench_duckdb()
bench_pyspark()


## Resultados e gráficos

In [ ]:
import matplotlib.pyplot as plt

resultados_df = pd.DataFrame(resultados)
resultados_df.to_csv(RESULTS_CSV, index=False)
print(f"Resultados salvos em: {RESULTS_CSV}")

if resultados_df.empty:
    print("Nenhum resultado registrado.")
else:
    mediana = (
        resultados_df.groupby(["framework", "operacao"], as_index=False)["tempo_s"]
        .median()
        .sort_values(["operacao", "framework"])
    )
    display(mediana)

    fig, ax = plt.subplots(figsize=(10, 4))
    mediana.pivot(index="operacao", columns="framework", values="tempo_s").plot(kind="bar", logy=True, ax=ax)
    ax.set_title("Tempo mediano por operacao")
    ax.set_xlabel("Operacao")
    ax.set_ylabel("Tempo mediano (s, escala log)")
    fig.tight_layout()
    tempo_png = BENCH_DIR / "tempos_medianos.png"
    fig.savefig(tempo_png, dpi=150)
    plt.close(fig)

    memoria = (
        resultados_df[resultados_df["operacao"] == "load_parquet"]
        .groupby("framework", as_index=False)["memoria_mb_delta"]
        .median()
    )
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(memoria["framework"], memoria["memoria_mb_delta"])
    ax.set_title("Variacao de memoria na carga")
    ax.set_ylabel("Delta RSS mediano (MB)")
    fig.tight_layout()
    memoria_png = BENCH_DIR / "memoria_carga.png"
    fig.savefig(memoria_png, dpi=150)
    plt.close(fig)

    print(f"Grafico de tempos:  {tempo_png}")
    print(f"Grafico de memoria: {memoria_png}")
